# TASK 3.2 — Classifier  (runs on `unmatched` only)

Two sub-models + agreement logic.

- **Sub-model A — XGBoost** via Model Serving, batch-scored on the 1024-dim
  `gte_embedding` (AUDIT A1). Numeric predictions are decoded to category
  names via the Phase 2 label index (AUDIT B5).
- **Sub-model B — `ai_query()`** zero-shot: the LLM picks one category from the
  full label list (ai_classify caps at 20 labels, so we use ai_query). Returns
  **only a label, no confidence**, so the only confidence in agreement is `xgboost_conf`.

Agreement → disagreements resolved by `ai_query()` GPT-4o with a **null-safe**
prompt and a valid label-array literal (AUDIT B3).

Output: `STAGE['classified']` with `final_category`, `ensemble_confidence`.

In [ ]:
%run ./config_notebook

In [ ]:
from pyspark.sql.functions import col, lit, when

unmatched = read_stage(spark, STAGE["unmatched"])
categories = load_categories(spark)
label_arr  = sql_label_array(categories)
decoder    = load_xgb_label_decoder(spark)
print(f"[3.2] unmatched rows: {unmatched.count():,}  categories: {len(categories)}")

### Sub-model A — XGBoost serving (batch, per-partition client)

In [ ]:
_endpoint = ENDPOINTS["xgboost"]
_feature  = FEATURE_COL          # "gte_embedding" (1024-dim dense array)
_decoder  = decoder              # {index: name}; empty → endpoint returns names


def _score_partition(iterator):
    from mlflow.deployments import get_deploy_client
    client = get_deploy_client("databricks")
    for pdf in iterator:
        vectors = [list(v) for v in pdf[_feature]]
        resp = client.predict(endpoint=_endpoint, inputs={"inputs": vectors})
        preds = resp["predictions"]
        names, confs = [], []
        for p in preds:
            # Endpoint may return {"category","confidence"} or {"prediction","probability"}
            raw = p.get("category", p.get("prediction"))
            conf = float(p.get("confidence", p.get("probability", 0.0)))
            if _decoder and not isinstance(raw, str):
                raw = _decoder.get(int(raw), str(raw))
            names.append(raw)
            confs.append(conf)
        pdf["xgboost_pred"] = names
        pdf["xgboost_conf"] = confs
        yield pdf


out_schema = unmatched.schema.add("xgboost_pred", "string").add("xgboost_conf", "double")
df = unmatched.mapInPandas(_score_partition, schema=out_schema)

### Sub-model B — `ai_query()` zero-shot (LLM picks one category; no label cap)

In [ ]:
df.createOrReplaceTempView("unmatched_temp")
# Sub-model B: LLM zero-shot. ai_classify() caps at 20 labels; we have many more
# categories, so we use ai_query() against the LLM endpoint (no label limit).
_llm = ENDPOINTS["llm"]
_label_csv = ", ".join(c.replace("'", "") for c in categories)
df = spark.sql(f"""
    SELECT *,
        ai_query('{_llm}', concat_ws(' ',
            'Classify this AP invoice line into exactly one category from the allowed list.',
            'Respond with ONLY the exact category name from the list, nothing else.',
            'Description:', COALESCE(final_description, ''),
            'Vendor:', COALESCE(vendor_name, ''),
            'Allowed categories:', '{_label_csv}')) AS llm_pred
    FROM unmatched_temp
""")

### Agreement + GPT-4o resolution
`xgboost_conf` is the only model confidence. Prompt uses `concat_ws` +
`COALESCE` so a NULL field never nullifies the whole prompt (AUDIT B3).

In [ ]:
high = PARAMS["agreement_high_conf"]
label_csv = ", ".join(c.replace("'", "") for c in categories)   # for the prompt text

df = (df
      .withColumn("agreement", col("xgboost_pred") == col("llm_pred"))
      .withColumn("final_category",
                  when(col("agreement"), col("xgboost_pred")).otherwise(lit(None)))
      .withColumn("ensemble_confidence",
                  when(col("agreement") & (col("xgboost_conf") > high), lit("HIGH"))
                  .when(col("agreement"), lit("MEDIUM"))
                  .otherwise(lit("RESOLVE"))))

df.createOrReplaceTempView("agreed_temp")
llm = ENDPOINTS["llm"]
resolved = spark.sql(f"""
    SELECT *,
        CASE WHEN ensemble_confidence = 'RESOLVE' THEN
            ai_query(
                '{llm}',
                concat_ws(' ',
                    'Pick the single best category for this AP invoice line.',
                    'XGBoost predicted:', COALESCE(xgboost_pred, 'n/a'),
                    '. LLM predicted:', COALESCE(llm_pred, 'n/a'),
                    '. Description:', COALESCE(final_description, ''),
                    '. Vendor:', COALESCE(vendor_name, ''),
                    '. Respond with exactly one label from this list:',
                    '{label_csv}'
                )
            )
        ELSE final_category END AS final_category_resolved
    FROM agreed_temp
""")

classified = (resolved
              .withColumn("final_category",
                          F.coalesce(col("final_category"), col("final_category_resolved")))
              .drop("final_category_resolved"))

In [ ]:
write_stage(classified, STAGE["classified"])
n_resolve = classified.filter(col("ensemble_confidence") == "RESOLVE").count()
print(f"[3.2] classified: {classified.count():,}  GPT-4o resolved: {n_resolve:,}")
dbutils.notebook.exit("3.2 OK")